# Ensaio — a tela do 2º codificador, sem gravar nada

Este notebook existe para **você** ver e experimentar o painel antes de entregá-lo
(ou para mostrá-lo a alguém). Ele carrega o mesmo runtime, o mesmo corpus e a mesma
evidência congelada do `04-revisao-assistida`, e monta o painel igual.

**A única diferença é o estado, e ele é descartável.** Duas travas: `offline=True`
faz o painel nascer sem servidor, e o cache vai para um arquivo temporário.
Responda, salve, avance, erre de propósito — nada entra no `coder2-data`, nada
conta como codificação e nada polui a segunda passada.

**Não codifique aqui.** O que você responder morre com a sessão do Colab. A
codificação que vale é a do `04-revisao-assistida`.

Uma consequência visível: o recibo embaixo do painel aparece **cinza**, dizendo
em que arquivo temporário a resposta caiu. No notebook real ele fica **verde** com
a hora quando a resposta chega ao servidor, e **vermelho** quando não chega — e
vermelho ali significa parar.


In [ ]:
#@title Instalação e configuração { display-mode: "form" }
# Roda uma vez por sessão. ~15 s. O SDK da Anthropic não entra: a evidência do
# modelo já vem congelada, e este notebook não chama modelo nenhum.
!pip -q install ipywidgets

import hashlib, json, sys, urllib.request
from pathlib import Path

SITE = "https://experimented-without-consent-corpus.vercel.app"

# O runtime vem do mesmo host do corpus, não de um clone: os repositórios são
# privados, e pedir um token dentro de uma célula seria pior que o problema que
# resolve. Cada arquivo é conferido contra o manifesto — isso pega download
# truncado e cópia velha, não é barreira contra adversário.
destino = Path("/content/ewc")
manifesto = json.load(urllib.request.urlopen(f"{SITE}/lib/manifest.json", timeout=60))
for arquivo in manifesto["arquivos"]:
    with urllib.request.urlopen(f"{SITE}/lib/{arquivo['file']}", timeout=60) as r:
        dados = r.read()
    if hashlib.sha256(dados).hexdigest() != arquivo["sha256"]:
        raise SystemExit(f"{arquivo['file']}: download não bate com o manifesto")
    alvo = destino / arquivo["file"]
    alvo.parent.mkdir(parents=True, exist_ok=True)
    alvo.write_bytes(dados)
print(f"runtime publicado em {manifesto['publicado_em'][:10]} · "
      f"{len(manifesto['arquivos'])} arquivos conferidos")

sys.path.insert(0, str(destino / "analysis"))
import revisao as R

# O corpus é lido do mesmo site. Para trabalhar offline, baixe `md/` e aponte
# CORPUS para a pasta local — o resto do notebook não muda.
corpus = R.configurar(corpus=f"{SITE}/md", modelo="claude-opus-5")
print(f"{sum(len(s['docs']) for s in corpus.index['services'])} documentos · "
      f"{len(corpus.index['services'])} serviços")

# A procedência aparece UMA vez, aqui, e vale para as 26 células abaixo: a data do
# congelamento e o país da captura são do corpus inteiro, não de cada serviço.
# Cada painel leva só o carimbo de uma linha.
R.mostrar_procedencia(corpus)


## O codebook, variável por variável

As dez variáveis com o critério completo: primeiro em linguagem direta, depois o
**texto exato do codebook congelado em 04/07/2026**, que é o que vale. Leia uma vez
antes de começar e volte aqui sempre que precisar.

Nas células de trabalho fica só o lembrete de uma linha, abaixo do título da
variável, e um ponteiro para cá. O critério inteiro repetido 26 vezes viraria
moldura — e o que a célula de trabalho precisa ser é uma tela de decisão.

As âncoras do piloto **não** entram aqui. Elas nomeiam serviços, e num bloco único
não há serviço em tela contra o qual suprimi-las: mostrá-las poria a resposta de um
serviço no seu campo de visão enquanto você codifica outro.


In [ ]:
# Mostra as dez variáveis com o critério completo. Nada de rede, nada de modelo.
R.mostrar_codebook()


In [ ]:
#@title Ensaio — escolha o serviço e rode { display-mode: "form" }
SERVICO = "Pinterest" #@param ["Wikipedia", "Temu", "Facebook", "YouTube", "Google Search", "WhatsApp", "Booking.com", "Google Maps", "Pornhub", "Amazon Store", "LinkedIn", "Bing", "XVideos", "Shein", "X", "XNXX", "Zalando", "Snapchat", "Stripchat", "App Store", "Instagram", "Google Play", "Pinterest", "Google Shopping", "TikTok", "AliExpress"]

import tempfile
from pathlib import Path
import coding_flow as F

# Duas travas, não uma. `configurar(offline=True)` faz o Estado padrão nascer
# sem servidor E corta qualquer chamada de modelo ao vivo — a evidência
# congelada continua sendo lida, que é justamente o que se quer ver. O Estado
# explícito manda o cache para um arquivo temporário, para não haver como
# confundir com o `coder2-local.json` de uma sessão de verdade.
R.configurar(offline=True)
ensaio = F.Estado(offline=True,
                  cache=Path(tempfile.mkdtemp(prefix="ensaio-")) / "estado.json")
print(f"ENSAIO · {SERVICO} · nada sai desta sessão ({ensaio.cache})")

R.painel(SERVICO, estado=ensaio)


## O que vale olhar enquanto você mexe

- **O cabeçalho**: quantos documentos o serviço tem, quantos são vinculantes, a
  data do congelamento e a vantagem (`IT`). É o que garante que os dois
  codificadores leram o mesmo texto.
- **As duas listas, separadas e com explicação.** Primeiro a busca por
  palavra-chave, com aviso onde o termo costuma dar falso positivo. Depois o que
  o modelo acrescentou. Nessa ordem ele só pode somar.
- **O portão.** Tente avançar sem a evidência ou sem o log de palavras-chave: a
  variável não fecha e a tela diz o que falta.
- **A ausência do botão de sugestão.** A evidência congelada traz citação e não
  sugestão de código, de propósito — quem atribui o código é o avaliador.
- **O recibo**, embaixo de tudo.
